In [9]:
import numpy as np
from scipy.sparse.linalg import spsolve
from scipy.interpolate import griddata

# Define your initial nodes and elements here
initial_nodes = np.array([[0.0, 0.0, 1],
                          [1.0, 0.0, 1],
                          [1.0, 1.0, 1],
                          [0.0, 1.0, 1],
                          [0.5798, 0.5986, 0]])  # Interior node (a, b)

# Elements should be defined with 0-based indexing
initial_elements = np.array([[0, 4, 3],  # Triangle 1
                             [4, 2, 3],  # Triangle 2
                             [0, 1, 4],  # Triangle 3
                             [1, 2, 4]])  # Triangle 4

# Define the exact solution function
def exact_solution(x1, x2):
    return x1 * (1 - x1) * np.exp(x1) * x2 * (1 - x2)

# Placeholder functions for mesh refinement and FEM computation
# These should be replaced with the actual implementations
def refine(nodes, elements):
    nnodes, dim = nodes.shape
    nelts, vertices_per_element = elements.shape

    newnodes = nodes.copy()
    newelements = np.zeros((4 * nelts, 3), dtype=int)
    inode = nnodes
    edgetable = {}

    for ie in range(nelts):
        newloc = np.zeros(3, dtype=int)

        for edge in range(3):
            n1 = elements[ie, edge]
            n2 = elements[ie, (edge + 1) % 3]
            edge_index = visited(edgetable, n1, n2)

            if edge_index == 0:
                mid_point = (nodes[n1, :2] + nodes[n2, :2]) / 2
                is_boundary = nodes[n1, 2] and nodes[n2, 2]
                newnode = np.array([[mid_point[0], mid_point[1], is_boundary]])
                newnodes = np.vstack([newnodes, newnode])
                newloc[edge] = inode
                edgetable[tuple(sorted((n1, n2)))] = inode
                inode += 1
            else:
                newloc[edge] = edge_index

        # Define new elements
        n0, n1, n2 = elements[ie, 0], elements[ie, 1], elements[ie, 2]
        m0, m1, m2 = newloc[0], newloc[1], newloc[2]
        newelements[4*ie] = [n0, m0, m2]
        newelements[4*ie+1] = [n1, m1, m0]
        newelements[4*ie+2] = [n2, m2, m1]
        newelements[4*ie+3] = [m0, m1, m2]

    return newnodes, newelements

def visited(edges, node1, node2):
    # Using a tuple of nodes as keys, where the tuple is always in a sorted order to handle bidirectionality
    edge_key = tuple(sorted((node1, node2)))

    # Check if the edge has already been visited by looking up the edge_key in the dictionary
    if edge_key in edges:
        return edges[edge_key]

    # Return 0 if the edge has not been visited
    return 0

def mu_area(X):
    # Takes three nodes of a triangle, listed as the rows
    # of a matrix X and finds the area of this triangle
    a = np.abs(np.linalg.det(np.column_stack((np.ones(3), X))) / 2)
    return a


def elt_stiffness(elements, nodes):
    nelts, m = elements.shape  # Number of elements

    elt_matrices = [None] * nelts

    for ie in range(nelts):
        elt_matrices[ie] = np.zeros((3, 3))

        # Get the coordinates of the three nodes of the element 'ie'
        # Corrected for zero-based indexing
        X = nodes[elements[ie], :2]

        # Calculate the area of the triangle
        area = mu_area(X)

        # Calculate edge vectors
        E = np.array([X[1] - X[2], X[2] - X[0], X[0] - X[1]])

        # Compute the stiffness matrix using the dot product of edge vectors
        Atau = np.zeros((3, 3))
        for i in range(3):
            for j in range(3):
                Atau[i, j] = np.dot(E[i], E[j])

        # Normalize by the factor of 4*area
        elt_matrices[ie] = Atau / (4 * area)

    return elt_matrices


from scipy.sparse import coo_matrix, csr_matrix

def global_stiffness(elt_matrices, elements, nodes):
    nelts, _ = elements.shape
    nnodes, _ = nodes.shape

    # Preparing data structures for COO format
    data = []
    row_indices = []
    col_indices = []

    # Assemble the global stiffness matrix
    for ie in range(nelts):
        iglob = elements[ie, :]  # Global indices of the current element's nodes
        for i in range(3):  # Assuming each element has 3 nodes (triangular elements)
            for j in range(3):
                data.append(elt_matrices[ie][i, j])
                row_indices.append(iglob[i])
                col_indices.append(iglob[j])

    # Create a COO matrix and convert it to CSR format
    Ahat = coo_matrix((data, (row_indices, col_indices)), shape=(nnodes, nnodes)).tocsr()

    return Ahat

def source_term(x1, x2):
    e = np.exp(1)  # Euler's number
    return -e**x1 * ((x1 + 3) * (x2 - 1) * x2 + 2 * (x1 - 1))
f_obj = source_term

# Define the exact solution given by the user
def exact_solution(x1, x2):
    return x1 * (1 - x1) * np.exp(x1) * x2 * (1 - x2)


def rhs(func, N0, elements, nodes):
    nelts, _ = elements.shape  # Number of elements
    nnodes, _ = nodes.shape    # Number of nodes

    fhat = np.zeros(nnodes)    # Initialize with a vector of zeros

    for ie in range(nelts):
        # Get the coordinates of the three nodes of element 'ie'
        X = nodes[elements[ie], :2]  # Zero-based indexing

        area = mu_area(X)  # Calculate the area of the element

        # Distribute the force contribution to each node
        for p in range(3):  # 3 nodes per element (assuming triangular elements)
            i = elements[ie, p]  # Zero-based node index
            fhat[i] += func(X[p, 0], X[p, 1]) * area / 3  # Function evaluated at each node

    return fhat[N0]  # Return only the non-Dirichlet part


# Define a function to run the simulation for a given number of refinements
def run_simulation(num_refinements, initial_nodes, initial_elements):
    nodes, elements = np.copy(initial_nodes), np.copy(initial_elements)
    # Uniform refinement of the mesh
    for i in range(num_refinements):
        nodes, elements = refine(nodes, elements)
    
    # Compute finite element solution
    elt_matrices = elt_stiffness(elements, nodes)
    Ahat = global_stiffness(elt_matrices, elements, nodes)
    N0 = np.where(nodes[:, 2] != 1)[0]
    A = Ahat[N0][:, N0]
    f = rhs(source_term, N0, elements, nodes)
    U = spsolve(A, f)
    
    # Extend the solution to include boundary conditions, if necessary
    Uhat = np.zeros(len(nodes))
    Uhat[N0] = U
    
    return Uhat, nodes, elements

# Define the number of refinements for each mesh size
num_refinements_list = [2, 3, 4, 5, 6]

# Define points to evaluate absolute error
points = [(0.5798, 0.5986), (0.92, 0.2)]

# Run the simulation for each mesh size and collect the results
for num_refinements in num_refinements_list:
    Uhat, nodes, elements = run_simulation(num_refinements, initial_nodes, initial_elements)
    
    # Interpolate the solution values at the specified points
    interpolated_values = griddata(nodes[:, :2], Uhat, points, method='linear')
    
    # Compute the exact solution at the specified points
    exact_values = [exact_solution(*pt) for pt in points]
    
    # Compute the absolute errors
    errors = np.abs(exact_values - interpolated_values)
    
    # Print the absolute errors
    print(f"Mesh refinement level {num_refinements}:")
    for pt, error in zip(points, errors):
        print(f"Absolute error at {pt}: {error:.6f}")
    print("-------------------------------------------------------")


Mesh refinement level 2:
Absolute error at (0.5798, 0.5986): 0.083058
Absolute error at (0.92, 0.2): 0.007347
-------------------------------------------------------
Mesh refinement level 3:
Absolute error at (0.5798, 0.5986): 0.087296
Absolute error at (0.92, 0.2): 0.011635
-------------------------------------------------------
Mesh refinement level 4:
Absolute error at (0.5798, 0.5986): 0.089047
Absolute error at (0.92, 0.2): 0.012195
-------------------------------------------------------
Mesh refinement level 5:
Absolute error at (0.5798, 0.5986): 0.089650
Absolute error at (0.92, 0.2): 0.012298
-------------------------------------------------------
Mesh refinement level 6:
Absolute error at (0.5798, 0.5986): 0.089842
Absolute error at (0.92, 0.2): 0.012318
-------------------------------------------------------


In [10]:
def run_simulation(num_refinements, initial_nodes, initial_elements):
    nodes, elements = np.copy(initial_nodes), np.copy(initial_elements)
    for i in range(num_refinements):
        nodes, elements = refine(nodes, elements)
    
    elt_matrices = elt_stiffness(elements, nodes)
    Ahat = global_stiffness(elt_matrices, elements, nodes)
    N0 = np.where(nodes[:, 2] != 1)[0]
    A = Ahat[N0][:, N0]
    f = rhs(exact_solution, N0, elements, nodes)
    U = spsolve(A, f)
    Uhat = np.zeros(len(nodes))
    Uhat[N0] = U
    return Uhat, nodes, elements

# Initialize the mesh
initial_nodes = np.array([[0.0, 0.0, 1],
                          [1.0, 0.0, 1],
                          [1.0, 1.0, 1],
                          [0.0, 1.0, 1],
                          [0.5798, 0.5986, 0]])  # Interior node (a, b)

initial_elements = np.array([[0, 4, 3],
                             [4, 2, 3],
                             [0, 1, 4],
                             [1, 2, 4]])

# Points to evaluate absolute error
points = [(0.5798, 0.5986), (0.92, 0.2)]

# Number of refinements for each mesh size
num_refinements_list = [2, 3, 4, 5, 6]

# Run the simulation for each mesh size and collect the results
for num_refinements in num_refinements_list:
    Uhat, nodes, elements = run_simulation(num_refinements, initial_nodes, initial_elements)
    interpolated_values = griddata(nodes[:, :2], Uhat, points, method='linear')
    exact_values = [exact_solution(*pt) for pt in points]
    errors = np.abs(exact_values - interpolated_values)
    
    # Print the absolute errors
    print(f"Mesh refinement level {num_refinements}:")
    for pt, error in zip(points, errors):
        print(f"Absolute error at {pt}: {error:.6f}")
    print("-------------------------------------------------------")

Mesh refinement level 2:
Absolute error at (0.5798, 0.5986): 0.099386
Absolute error at (0.92, 0.2): 0.028627
-------------------------------------------------------
Mesh refinement level 3:
Absolute error at (0.5798, 0.5986): 0.099278
Absolute error at (0.92, 0.2): 0.028550
-------------------------------------------------------
Mesh refinement level 4:
Absolute error at (0.5798, 0.5986): 0.099228
Absolute error at (0.92, 0.2): 0.028547
-------------------------------------------------------
Mesh refinement level 5:
Absolute error at (0.5798, 0.5986): 0.099209
Absolute error at (0.92, 0.2): 0.028550
-------------------------------------------------------
Mesh refinement level 6:
Absolute error at (0.5798, 0.5986): 0.099203
Absolute error at (0.92, 0.2): 0.028549
-------------------------------------------------------


In [1]:
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.interpolate import griddata

def exact_solution(x1, x2):
    """Exact solution for comparison."""
    return x1 * (1 - x1) * np.exp(x1) * x2 * (1 - x2)

def refine(nodes, elements):
    """Refines the mesh by splitting each triangle into four smaller triangles."""
    new_nodes = np.vstack([nodes])
    new_elements = []
    edge_midpoints = {}  # To avoid duplicating midpoints
    
    for element in elements:
        indices = list(element)
        midpoints = []
        
        # Create midpoints for each edge and store new indices
        for i in range(3):
            edge = tuple(sorted((indices[i], indices[(i + 1) % 3])))
            if edge not in edge_midpoints:
                midpoint = (nodes[edge[0]] + nodes[edge[1]]) / 2
                midpoint[2] = nodes[edge[0], 2] and nodes[edge[1], 2]  # Preserve boundary info
                new_nodes = np.vstack([new_nodes, midpoint])
                edge_midpoints[edge] = len(new_nodes) - 1
            midpoints.append(edge_midpoints[edge])
        
        # Create four new triangles from the old triangle
        a, b, c = indices
        ab, bc, ca = midpoints
        new_elements.extend([
            [a, ab, ca],
            [ab, b, bc],
            [ca, bc, c],
            [ab, bc, ca]
        ])
        
    return new_nodes, np.array(new_elements, dtype=int)

def element_stiffness(node_positions):
    """Calculate element stiffness matrix using the area method for triangular elements."""
    a = node_positions[0]
    b = node_positions[1]
    c = node_positions[2]
    area = 0.5 * np.abs(a[0] * (b[1] - c[1]) + b[0] * (c[1] - a[1]) + c[0] * (a[1] - b[1]))
    # Simplistic stiffness matrix proportional to the area of the triangle
    return (1 / (4 * area)) * np.ones((3, 3))  # Placeholder

def global_stiffness(nodes, elements):
    """Assembles the global stiffness matrix."""
    size = len(nodes)
    matrix_entries = []
    row_ind = []
    col_ind = []
    
    for element in elements:
        node_indices = element[:3]
        K_e = element_stiffness(nodes[node_indices, :2])
        
        for i in range(3):
            for j in range(3):
                matrix_entries.append(K_e[i, j])
                row_ind.append(node_indices[i])
                col_ind.append(node_indices[j])
                
    A = coo_matrix((matrix_entries, (row_ind, col_ind)), shape=(size, size))
    return A.tocsr()

def solve_fem(nodes, elements):
    """Solves the FEM system."""
    A = global_stiffness(nodes, elements)
    b = np.zeros(len(nodes))  # Assuming no external forces
    u = spsolve(A, b)
    return u

# Initialization
nodes = np.array([[0.0, 0.0, 1], [1.0, 0.0, 1], [1.0, 1.0, 1], [0.0, 1.0, 1], [0.5798, 0.5986, 0]])
elements = np.array([[0, 4, 3], [4, 2, 3], [0, 1, 4], [1, 2, 4]])

# Perform refinements and solve
for i in range(1, 6):  # 5 levels of refinement
    nodes, elements = refine(nodes, elements)
    u = solve_fem(nodes, elements)
    print(f"Mesh refinement level {i}:")
    for point in [(0.5798, 0.5986), (0.92, 0.2)]:
        value = griddata(nodes[:, :2], u, np.array([point]), method='linear')
        exact = exact_solution(*point)
        error = np.abs(exact - value)
        print(f"Error at {point}: {error[0]}")


Mesh refinement level 1:
Error at (0.5798, 0.5986): 0.10453272703389084
Error at (0.92, 0.2): 0.029549403631889827
Mesh refinement level 2:
Error at (0.5798, 0.5986): nan
Error at (0.92, 0.2): nan
Mesh refinement level 3:
Error at (0.5798, 0.5986): 0.10453272703389084
Error at (0.92, 0.2): 0.029549403631889827
Mesh refinement level 4:
Error at (0.5798, 0.5986): 0.10453272703389084
Error at (0.92, 0.2): 0.029549403631889827


/opt/conda/lib/python3.9/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:293: MatrixRankWarning: Matrix is exactly singular
  warn("Matrix is exactly singular", MatrixRankWarning)


Mesh refinement level 5:
Error at (0.5798, 0.5986): 0.10453272703389084
Error at (0.92, 0.2): 0.029549403631889827
